
# Appendix 01: Special Functions

Source span: printed pages 349-352; PDF pages 362-365. The source was inspected for orientation only: it identifies which special-function identities the course needs later, but the explanations, examples, plots, and checks here are original.

## Chapter Goal

This appendix is the numerical service room for the directional-statistics course. The earlier chapters use circular and spherical probability models, and those models need normalizing constants, mean-resultant equations, derivative ratios, and table replacements. The goal here is to make those special functions visible enough that a reader can tell when a formula is behaving as geometry says it should.

The central habit is simple: a normalizing function is not just a symbol in the margin. It tells us how mass is distributed over a curved sample space. A ratio of Bessel functions turns concentration into an expected resultant length. A derivative of a Kummer function turns an axial concentration parameter into a moment. A hypergeometric integral over a sphere or a Stiefel manifold says that the calculation is averaging over directions, frames, or axes rather than over an ordinary line.

The notebook therefore treats the appendix as a set of executable calculators. Each section names the function, draws the quantity that must be inspected, and records a numerical invariant. The artifacts are book-local and concept-named: a Bessel-ratio atlas, a Kummer derivative-ratio lab, a sphere-integral check, and an interactive surface linking dimension, concentration, and expected resultant length.



## Visualization Storyboard And Library Routing

The source span is compact, but it supports several recurring computations in the course. The visualization plan is to separate four questions that otherwise get compressed into a list of formulas.

- **Bessel-ratio atlas.** The modified Bessel ratio
  \[
  A_p(kappa)=\frac{I_{p/2}(kappa)}{I_{p/2-1}(kappa)}
  \]
  is the bridge between a von Mises-Fisher concentration and the expected resultant length in ambient dimension `p`. A Matplotlib atlas is appropriate because the reader needs stable labeled curves and error bands that will still be readable in a static export. The inspection target is monotonicity and the way dimension changes the same concentration value.

- **Executable asymptotics.** The appendix lists small- and large-concentration expansions. Instead of asking the reader to trust a printed approximation, the notebook plots relative error. The inspection target is the regime where a cheap approximation becomes safe enough for estimation.

- **Kummer derivative-ratio lab.** Axial models use the confluent hypergeometric function `M(a,b,kappa)`. A useful quantity is the derivative of `log M`, which can be computed from the identity `dM/dkappa = (a/b) M(a+1,b+1,kappa)`. A second static artifact compares the exact ratio to its small- and large-concentration approximations.

- **Sphere/Stiefel integral check.** The matrix-argument hypergeometric formulas are conceptually integrals over frames. For the one-frame case, the Stiefel manifold is just the unit sphere. A deterministic sphere quadrature checks that the Bessel normalizer agrees with the direct average of `exp(kappa x_1)`. The inspection target is that a normalizing constant is an average over the curved sample space, not a lookup-table magic number.

Plotly is used only where interaction adds value: a surface over `dimension x concentration` lets the reader see that `A_p` and the Kummer ratio have similar limiting behavior but different low-concentration geometry. SciPy supplies the special functions; NumPy supplies deterministic grids; Matplotlib supplies durable audit-friendly PNGs.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "appendix-01"
source_span = {"printed": "349-352", "pdf": "362-365"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import optimize, special

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def bessel_A(p: float, kappa):
    """Mean-resultant ratio I_{p/2}(kappa) / I_{p/2-1}(kappa)."""
    kappa = np.asarray(kappa, dtype=float)
    out = np.zeros_like(kappa, dtype=float)
    mask = kappa > 1e-12
    v = p / 2.0
    if np.any(mask):
        out[mask] = special.ive(v, kappa[mask]) / special.ive(v - 1.0, kappa[mask])
    return out if out.shape else float(out)


def bessel_A_small(p: float, kappa):
    kappa = np.asarray(kappa, dtype=float)
    return kappa / p - (kappa**3) / (p**2 * (p + 2.0))


def bessel_A_large(p: float, kappa):
    kappa = np.asarray(kappa, dtype=float)
    return 1.0 - (p - 1.0) / (2.0 * kappa)


def invert_bessel_A(p: float, resultant: float) -> float:
    if resultant <= 1e-12:
        return 0.0
    if not 0.0 < resultant < 1.0:
        raise ValueError("resultant must lie in (0, 1)")
    upper = max(10.0, (p - 1.0) / max(1e-4, 2.0 * (1.0 - resultant)))
    f = lambda value: bessel_A(p, np.array([value]))[0] - resultant
    while f(upper) < 0:
        upper *= 2.0
    return float(optimize.brentq(f, 1e-10, upper, maxiter=100))


def vmf_normalizer(p: float, kappa):
    kappa = np.asarray(kappa, dtype=float)
    nu = p / 2.0 - 1.0
    area = 2.0 * np.pi ** (p / 2.0) / special.gamma(p / 2.0)
    out = np.empty_like(kappa, dtype=float)
    small = kappa <= 1e-12
    out[small] = 1.0 / area
    if np.any(~small):
        kk = kappa[~small]
        out[~small] = kk**nu / ((2.0 * np.pi) ** (p / 2.0) * special.iv(nu, kk))
    return out if out.shape else float(out)


def kummer_D(p: float, kappa):
    """Derivative ratio d/dkappa log M(1/2, p/2, kappa)."""
    kappa = np.asarray(kappa, dtype=float)
    a = 0.5
    b = p / 2.0
    numerator = (a / b) * special.hyp1f1(a + 1.0, b + 1.0, kappa)
    denominator = special.hyp1f1(a, b, kappa)
    out = numerator / denominator
    return out if out.shape else float(out)


def kummer_D_small(p: float, kappa):
    kappa = np.asarray(kappa, dtype=float)
    return 1.0 / p + (2.0 * (p - 1.0) / (p**2 * (p + 2.0))) * kappa


def kummer_D_large(p: float, kappa):
    kappa = np.asarray(kappa, dtype=float)
    return 1.0 - (p - 1.0) / (2.0 * kappa)


def sphere_points(n: int = 40000):
    i = np.arange(n, dtype=float)
    z = 1.0 - 2.0 * (i + 0.5) / n
    theta = np.pi * (3.0 - np.sqrt(5.0)) * i
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(theta), r * np.sin(theta), z])


def sphere_mgf_s2(kappa):
    kappa = np.asarray(kappa, dtype=float)
    out = np.ones_like(kappa, dtype=float)
    mask = np.abs(kappa) > 1e-12
    out[mask] = np.sinh(kappa[mask]) / kappa[mask]
    return out if out.shape else float(out)



## Modified Bessel Ratios As Resultant Calculators

For a concentrated sample on a circle or sphere, the resultant length is the geometric summary that says how strongly the observations point in a common direction. The modified Bessel ratio `A_p(kappa)` predicts that summary under a von Mises-Fisher model. This turns an estimation problem into an inversion problem: observe a mean resultant length, then solve `A_p(kappa) = Rbar`.

The figure below teaches three facts that are easy to miss when the appendix is read as a list. First, the ratio is monotone, so the inversion is well posed. Second, dimension matters: the same `kappa` produces a smaller expected resultant in higher ambient dimension because there are more directions in which probability can spread. Third, the small- and large-concentration approximations should be used as local tools, not universal substitutes.

The lower panels make the table-replacement idea explicit. Instead of consulting a fixed row, the reader can compute a concentration from any admissible `Rbar` and inspect the normalizing constant that would accompany the corresponding density. This is the bridge from an appendix formula to chapters on estimation and inference.


In [ ]:

k_grid = np.linspace(0.001, 14.0, 500)
p_values = [2, 3, 5, 10]
colors = plt.cm.viridis(np.linspace(0.12, 0.88, len(p_values)))

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
ax = axes[0, 0]
for p, color in zip(p_values, colors):
    ax.plot(k_grid, bessel_A(p, k_grid), color=color, label=f"p={p}")
ax.set_title("Bessel ratio A_p(kappa)")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("expected resultant length")
ax.legend(title="ambient dimension")
ax.set_ylim(0, 1.02)

ax = axes[0, 1]
for p, color in zip(p_values, colors):
    exact = bessel_A(p, k_grid)
    small = bessel_A_small(p, k_grid)
    large = bessel_A_large(p, k_grid)
    small_err = np.abs(small - exact)
    large_err = np.abs(large - exact)
    ax.semilogy(k_grid, small_err, color=color, linestyle="-", alpha=0.85, label=f"small p={p}")
    ax.semilogy(k_grid, large_err, color=color, linestyle="--", alpha=0.85, label=f"large p={p}")
ax.set_title("Approximation error: small solid, large dashed")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("absolute error")
ax.set_ylim(1e-6, 2)

ax = axes[1, 0]
r_grid = np.linspace(0.05, 0.95, 19)
for p, color in zip([2, 3, 5], colors[:3]):
    inv = np.array([invert_bessel_A(p, r) for r in r_grid])
    ax.plot(r_grid, inv, marker="o", color=color, label=f"p={p}")
ax.set_title("Executable inverse concentration table")
ax.set_xlabel("observed mean resultant Rbar")
ax.set_ylabel("solved kappa")
ax.legend()

ax = axes[1, 1]
for p, color in zip([2, 3, 5], colors[:3]):
    ax.plot(k_grid, np.log(vmf_normalizer(p, k_grid)), color=color, label=f"p={p}")
ax.set_title("Log normalizing constant for vMF densities")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("log c_p(kappa)")
ax.legend()

fig.suptitle("Appendix 01 calculator: Bessel ratios, approximations, inverse lookup, and normalizers", y=1.02, fontsize=14)
fig.tight_layout()
ratio_path = save_matplotlib(fig, TOPIC, "bessel-ratios", "bessel-ratio-and-asymptotics.png")
plt.close(fig)
display_artifact(ratio_path, width=920)

ratio_diagnostics = {
    f"A_p_monotone_p{p}": bool(np.all(np.diff(bessel_A(p, k_grid)) > 0))
    for p in p_values
}
ratio_diagnostics["inverse_roundtrip_max_abs_error"] = float(
    max(abs(bessel_A(3, np.array([invert_bessel_A(3, r)]))[0] - r) for r in r_grid)
)
ratio_diagnostics["normalizers_positive"] = bool(np.all(vmf_normalizer(3, k_grid) > 0))
ratio_diagnostics



## Kummer Functions For Axial Moments

The Bessel ratio is not the only derivative-ratio object in the course. Axial models and several spherical tests use the Kummer function `M(a,b,kappa)`, also called the confluent hypergeometric function. The useful computational object is often

\[
D_p(kappa)=\frac{d}{d kappa}\log M(1/2,p/2,kappa).
\]

This ratio has a direct interpretation: it is the expected value of a squared axial projection under a tilted axial density. At zero concentration every coordinate direction is equally likely, so the baseline is `1/p`. As concentration grows, the favored axis receives more mass and the ratio approaches one. That is the same visual story as a cluster tightening on a sphere, but now it is an axial second-moment story rather than a directional first-moment story.

The derivative identity is a good audit hook. If the implementation computes `D_p` by the ratio `(1/p) M(3/2,p/2+1,kappa) / M(1/2,p/2,kappa)`, then a finite-difference derivative of `log M` should agree. The plot and the JSON checks below make that identity executable.


In [ ]:

k_kummer = np.linspace(0.001, 12.0, 450)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
for p, color in zip(p_values, colors):
    ax.plot(k_kummer, kummer_D(p, k_kummer), color=color, label=f"p={p}")
    ax.plot(k_kummer, kummer_D_small(p, k_kummer), color=color, linestyle=":", alpha=0.55)
ax.set_title("Kummer derivative ratio D_p(kappa)")
ax.set_xlabel("axial concentration kappa")
ax.set_ylabel("d log M / d kappa")
ax.set_ylim(0, 1.02)
ax.legend(title="solid exact, dotted small-k")

ax = axes[1]
for p, color in zip(p_values, colors):
    exact = kummer_D(p, k_kummer)
    large = kummer_D_large(p, k_kummer)
    ax.semilogy(k_kummer, np.abs(large - exact), color=color, label=f"p={p}")
ax.set_title("Large-concentration approximation error")
ax.set_xlabel("axial concentration kappa")
ax.set_ylabel("absolute error")
ax.set_ylim(1e-5, 2)
ax.legend()

fig.suptitle("Kummer normalizer lab: derivative ratios and asymptotic regimes", y=1.04, fontsize=14)
fig.tight_layout()
kummer_path = save_matplotlib(fig, TOPIC, "kummer-ratios", "kummer-derivative-ratio-lab.png")
plt.close(fig)
display_artifact(kummer_path, width=920)

fd_points = np.array([0.2, 1.0, 4.0, 8.0])
fd_errors = []
step = 1e-5
for p in [2, 3, 5, 8]:
    b = p / 2.0
    logM_plus = np.log(special.hyp1f1(0.5, b, fd_points + step))
    logM_minus = np.log(special.hyp1f1(0.5, b, fd_points - step))
    finite_diff = (logM_plus - logM_minus) / (2.0 * step)
    exact = kummer_D(p, fd_points)
    fd_errors.extend(np.abs(finite_diff - exact).tolist())

kummer_diagnostics = {
    "D_p_bounds_hold": bool(np.all((kummer_D(5, k_kummer) > 0) & (kummer_D(5, k_kummer) < 1))),
    "D_p_starts_near_one_over_p": float(abs(kummer_D(5, np.array([1e-5]))[0] - 1 / 5)),
    "finite_difference_max_abs_error": float(max(fd_errors)),
}
kummer_diagnostics



## One-Frame Stiefel Integral As A Sphere Check

The matrix-argument hypergeometric formulas in the appendix can feel remote because they live over Stiefel manifolds. The smallest nontrivial case is friendly: a one-frame in three-dimensional space is a point on the sphere. The integral of `exp(kappa x_1)` over the uniformly distributed unit sphere is therefore a direct normalizer check.

For `S^2`, the average is `sinh(kappa)/kappa`. The code below does not use random sampling. It uses a deterministic Fibonacci sphere grid, evaluates `exp(kappa x_1)`, and compares the numerical average with the Bessel/hypergeometric expression. This makes the hidden geometry of the formula visible: the special function is what remains after all possible directions have been averaged.

The visual also explains why high concentration is numerically delicate. As `kappa` grows, most of the contribution comes from a small cap near the preferred direction. A bad quadrature or a casual floating-point implementation can then underestimate the normalizer. Later chapters that fit spherical models should keep this cap-concentration picture in mind.


In [ ]:

points = sphere_points(60000)
probe_k = np.linspace(0.0, 5.0, 26)
quad = np.array([np.exp(k * points[:, 2]).mean() for k in probe_k])
closed = sphere_mgf_s2(probe_k)
rel_error = np.abs(quad - closed) / np.maximum(1.0, np.abs(closed))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(probe_k, closed, label="closed form sinh(k)/k", color="#345995")
axes[0].scatter(probe_k, quad, label="Fibonacci sphere average", color="#E07A5F", s=28, zorder=3)
axes[0].set_title("Sphere integral normalizer check")
axes[0].set_xlabel("concentration kappa")
axes[0].set_ylabel("uniform sphere average")
axes[0].legend()

axes[1].semilogy(probe_k[1:], rel_error[1:], marker="o", color="#6A994E")
axes[1].set_title("Relative quadrature error")
axes[1].set_xlabel("concentration kappa")
axes[1].set_ylabel("relative error")
axes[1].set_ylim(1e-9, 1e-2)

fig.suptitle("One-frame Stiefel case: averaging an exponential over the sphere", y=1.04, fontsize=14)
fig.tight_layout()
stiefel_path = save_matplotlib(fig, TOPIC, "stiefel-integrals", "sphere-integral-normalizer-check.png")
plt.close(fig)
display_artifact(stiefel_path, width=920)

stiefel_diagnostics = {
    "sphere_grid_point_count": int(points.shape[0]),
    "sphere_points_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(points, axis=1) - 1.0))),
    "sphere_integral_max_relative_error": float(np.max(rel_error)),
    "sphere_integral_passes_tolerance": bool(np.max(rel_error) < 2e-4),
}
stiefel_diagnostics



## Interactive Surface: Dimension Changes The Calculator

A printed appendix can list identities, but it is hard to see a family of functions move with dimension from a printed line. The interactive artifact below plots two surfaces over `(p, kappa)`: the Bessel resultant ratio `A_p` and the Kummer derivative ratio `D_p`. They both begin at a dimension-dependent baseline and both approach one in the high-concentration limit, but they answer different statistical questions.

Use the surface to inspect three regimes. Near `kappa = 0`, `A_p` starts at zero because a directional first moment vanishes under uniformity, while `D_p` starts at `1/p` because a squared axial projection still has an average size. In the middle regime, the surfaces separate most clearly and table interpolation is least trustworthy. In the high-concentration regime, both ratios approach one, and the large-concentration approximations become useful.


In [ ]:

p_surface = np.arange(2, 13, dtype=float)
k_surface = np.linspace(0.001, 14.0, 180)
K, P = np.meshgrid(k_surface, p_surface)
A_surface = np.vstack([bessel_A(p, k_surface) for p in p_surface])
D_surface = np.vstack([kummer_D(p, k_surface) for p in p_surface])

surface_fig = go.Figure()
surface_fig.add_trace(go.Surface(x=K, y=P, z=A_surface, colorscale="Viridis", opacity=0.86, name="A_p", showscale=True, colorbar=dict(title="A_p")))
surface_fig.add_trace(go.Surface(x=K, y=P, z=D_surface, colorscale="Reds", opacity=0.55, name="D_p", showscale=False))
surface_fig.update_layout(
    title="Appendix 01 interactive calculator surface: Bessel A_p and Kummer D_p",
    scene=dict(
        xaxis_title="concentration kappa",
        yaxis_title="ambient dimension p",
        zaxis_title="ratio value",
        zaxis=dict(range=[0, 1.02]),
    ),
    margin=dict(l=0, r=0, t=48, b=0),
    height=620,
)
surface_path = save_plotly_html(surface_fig, TOPIC, "interactive", "bessel-kummer-ratio-surface.html", include_plotlyjs=True)
display_artifact(surface_path, width="100%", height=640)

surface_diagnostics = {
    "A_surface_min": float(A_surface.min()),
    "A_surface_max": float(A_surface.max()),
    "D_surface_min": float(D_surface.min()),
    "D_surface_max": float(D_surface.max()),
    "surface_values_in_unit_interval": bool((A_surface.min() >= 0) and (A_surface.max() <= 1.0) and (D_surface.min() >= 0) and (D_surface.max() <= 1.0)),
}
surface_diagnostics



## Table Replacement Lab

The appendix supports later tables by giving formulas that can be evaluated directly. A good table replacement should expose three things: the input statistic, the solved parameter, and the residual that proves the lookup is internally consistent. The small lab below solves `A_p(kappa)=Rbar` for a few representative resultant lengths and records the round-trip residual.

The point is not that every reader should memorize the numbers. The point is that the executable version has a visible domain. `Rbar` must lie between zero and one; values near one require very large concentration; and high-dimensional samples require more concentration to reach the same resultant length. These are geometric facts about spheres, not quirks of SciPy.

The final JSON file stores the table replacement beside the artifact checks. It is intentionally plain so that an audit script or a human reader can see which invariants were tested: monotonicity, inverse round trips, derivative identity, integral agreement, surface bounds, positive normalizers, and nonzero artifact sizes.


In [ ]:

lookup_rows = []
for p in [2, 3, 5, 10]:
    for r in [0.25, 0.50, 0.75, 0.90]:
        solved = invert_bessel_A(p, r)
        roundtrip = bessel_A(p, np.array([solved]))[0]
        lookup_rows.append({
            "p": int(p),
            "Rbar": float(r),
            "kappa_hat": float(solved),
            "roundtrip_abs_error": float(abs(roundtrip - r)),
        })

numeric_checks = {
    **ratio_diagnostics,
    **kummer_diagnostics,
    **stiefel_diagnostics,
    **surface_diagnostics,
    "lookup_rows": lookup_rows,
    "lookup_max_roundtrip_abs_error": float(max(row["roundtrip_abs_error"] for row in lookup_rows)),
    "all_boolean_checks_pass": bool(
        all(value for value in ratio_diagnostics.values() if isinstance(value, bool))
        and all(value for value in kummer_diagnostics.values() if isinstance(value, bool))
        and all(value for value in stiefel_diagnostics.values() if isinstance(value, bool))
        and all(value for value in surface_diagnostics.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert numeric_checks["lookup_max_roundtrip_abs_error"] < 1e-9
assert numeric_checks["finite_difference_max_abs_error"] < 1e-8
assert numeric_checks["sphere_integral_max_relative_error"] < 2e-4

checks_path = save_json(numeric_checks, TOPIC, "checks", "special-function-calculator-checks.json")
display_artifact(checks_path)
numeric_checks



## Takeaways

This appendix becomes much more useful when read as a set of geometric calculators.

- `A_p(kappa)` is an estimation bridge. It converts concentration into expected resultant length, and its monotonicity makes inverse concentration lookup possible.
- Small- and large-concentration approximations are regime statements. They are valuable only when their error is inspected beside the exact special function.
- The Kummer derivative ratio `D_p(kappa)` is the axial analogue of a moment calculator. It starts at `1/p`, not zero, because a squared projection has a uniform baseline even when there is no preferred axis.
- Hypergeometric functions of matrix argument encode averages over curved spaces such as spheres and Stiefel manifolds. The one-frame sphere check is the smallest case of that idea.
- The course should prefer executable tables over copied tables. A notebook can show the admissible domain, solve an inverse problem, record the residual, and preserve the artifact trail.

A later chapter can now cite this appendix operationally: if it estimates concentration, it should say which ratio is being inverted; if it uses an approximation, it should name the inspected error regime; if it invokes a matrix-argument normalizer, it should identify the sample space being averaged over. That is what makes the appendix stand alone rather than merely repeat a formula list.


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [ratio_path, kummer_path, stiefel_path, surface_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "bessel_ratio_monotone_all_plotted_dimensions": all(
            ratio_diagnostics[f"A_p_monotone_p{p}"] for p in p_values
        ),
        "bessel_inverse_roundtrip_under_tolerance": ratio_diagnostics["inverse_roundtrip_max_abs_error"] < 1e-9,
        "kummer_derivative_identity_under_tolerance": kummer_diagnostics["finite_difference_max_abs_error"] < 1e-8,
        "sphere_integral_under_tolerance": stiefel_diagnostics["sphere_integral_max_relative_error"] < 2e-4,
        "interactive_surface_bounds_valid": surface_diagnostics["surface_values_in_unit_interval"],
    },
    "pdf_used_for": "source orientation only; no copied prose, tables, screenshots, page crops, or figures",
    "standalone_contract": "direct explanations, concept-named artifacts, executable calculators, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 100
final_sanity
